[오늘의 미션] 3/13

wconcept > SQL 활용 다음과 같은 값을 조회.출력

카테고리별(의류, 가방, 신발, 액세서리) 평균 판매가, 평균 할인율

1. 총 9개의 카테고리의 버튼을 클릭하고 그 안에 값을 크롤링한다
- 버튼을 누르고 값을 조회하면 다시 버튼을 눌러야하니 반복문을 사용해야한다
- 카테고리별로 구분해야해야한다
  
2. 값은 소비자가 / 판매가 / 할인률 / 브랜드 / 상품명만 가지고 온다
- brand_name, product_name, original_price, sale_price, discount_rate

3. 가져온 값을 pymysql을 통해 mysql로 보낸다
- 테이블은 브랜드 전용 테이블, 카테고리 전용 테이블, 상품 전용 테이블 3개를 만든다
- 각각의 테이블엔 id값이 들어가야하고 서로 join으로 연결할 수 있도록 id값을 FK로 설정해야한다

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

import re
import time
import pymysql

#mysql 접속정보
conn = pymysql.connect(
    host="127.0.0.1",
    user="root",
    password="ksdir8558",
    db="wconsept_db2",
    charset="utf8mb4"
)

cursor =  conn.cursor()

service = Service(ChromeDriverManager().install())
options = Options()

# options.add_argument("--headless")
# options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36")
options.add_argument("--lang=ko=KR")
options.add_argument("--no-sandbox")

target_url = "https://display.wconcept.co.kr/rn/women"

driver = webdriver.Chrome(service=service, options=options)
driver.get(target_url)
time.sleep(3)

best_btn = driver.find_element(By.XPATH, "//span[text() = '베스트']")
best_btn.click()
time.sleep(3)

# bag_btn = driver.find_element(By.XPATH, "//div[data-value() = '10102']")
# bag_btn.click()

items = driver.find_elements(By.CSS_SELECTOR, "div.product-item.item.type-all")
print("찾은 상품 수:", len(items))

def to_int(text) :
    num = re.sub(r"[^0-9]", "", text)
    return int(num) if num else 0

for idx, item in enumerate(items, 1) :
    #브랜드 & 상품 정보
    brand_and_product = item.find_element(By.CSS_SELECTOR, "span.prdc-title")
    brand_name = brand_and_product.find_element(By.CSS_SELECTOR, "span.text.title").text.strip()
    product_name = brand_and_product.find_element(By.CSS_SELECTOR, "span.text.detail").text.strip()
    
    
    #소비자가 & 판매가 & 할인률 정보
    price_box = item.find_element(By.CSS_SELECTOR, "span.prdc-price")

    try :
        original_price = to_int (price_box.find_element(By.CSS_SELECTOR, "span.customer-price").text.strip())
    except : 
        original_price = 0

    try :
        sale_price = to_int (price_box.find_element(By.CSS_SELECTOR, "span.final-price").text.strip())
    except : 
        sale_price = 0

    try :
        discount_rate = to_int (price_box.find_element(By.CSS_SELECTOR, "span.final-discount").text.strip())
    except : 
        discount_rate = 0

    
    #mysql 테이블 저장
    cursor.execute("""
        INSERT INTO brands (brand_name) VALUES (%s) ON DUPLICATE KEY UPDATE brand_name = VALUES(brand_name)
    """, (brand_name,))

    cursor.execute("SELECT brand_id FROM brands WHERE brand_name = %s", (brand_name,))
    brand_id = cursor.fetchone()[0]
    
    cursor.execute(""" 
        INSERT INTO products (brand_id, rank_no, product_name, original_price, 
        sale_price, discount_rate, rating, review_count, like_count, product_url
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s) 
    """, (brand_id, idx, product_name, original_price, sale_price, discount_rate, rating, review_count, like_count, product_url))

    print(f"{idx}위 저장 완료: {brand_name} / {product_name} / {product_no}")

conn.commit()
conn.close()

    
driver.quit()